# PA4 Task 1 — Kaggle launcher (P3/Q3)
Thin launcher, no experiment logic. Settings: **Accelerator GPU T4 x2**, **Internet on**, input = your private dataset
with the 7 `*_locked.jsonl.gz` files + `local_source_revisions.json`.
Run with **Save Version → Save & Run All (Commit)**; results are in the version's Output tab (`results/`, `task1_splits/`).
GPU 0: sensitivity, then P3 50% (dev sweep → eval) and Q3 INT4 budget. GPU 1: P3 75% and Q3 INT3 budget.

In [ ]:
BRANCH= "task1-compress"
EVALS= "wikitext2 c4_val hellaswag resources"
NAMES= ["p3_nonuniform_s0.5", "p3_nonuniform_s0.75", "q3_mixed_budget_b4", "q3_mixed_budget_b3"]
import os
os.environ["BRANCH"]= BRANCH
os.environ["EVALS"]= EVALS

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=index,name,memory.total --format=csv
mkdir -p /tmp/pa4 && cd /tmp/pa4
rm -rf atml-pa4 atml-pa4-datasets
git clone -q -b "$BRANCH" https://github.com/Zeeba1705/atml-pa4.git
git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git
pip install -q transformers==4.57.1 huggingface_hub==0.36.2 "datasets>=3,<5" pyyaml

#locate the uploaded data, whatever the dataset slug; kaggle decompresses .gz uploads, so accept .jsonl too
HIT=$(find -L /kaggle/input \( -name 'c4_calibration_locked.jsonl.gz' -o -name 'c4_calibration_locked.jsonl' \) | head -1)
LOCK=$(find -L /kaggle/input -name local_source_revisions.json | head -1)
if [ -z "$HIT" ] || [ -z "$LOCK" ]; then
  echo "DATA NOT FOUND under /kaggle/input: add your dataset via Add Input (8 files incl. local_source_revisions.json)"
  find -L /kaggle/input -maxdepth 4 | head -50
  exit 1
fi
SRC=$(dirname "$HIT")
echo "data from $SRC"
mkdir -p atml-pa4-datasets/data/public_locked
for f in c4_calibration c4_development c4_validation_windows wikitext2_test_windows hellaswag_val gsm8k_train gsm8k_test; do
  dst=atml-pa4-datasets/data/public_locked/${f}_locked.jsonl.gz
  if [ -f "$SRC/${f}_locked.jsonl.gz" ]; then cp "$SRC/${f}_locked.jsonl.gz" "$dst"
  elif [ -f "$SRC/${f}_locked.jsonl" ]; then gzip -c "$SRC/${f}_locked.jsonl" > "$dst"
  else echo "MISSING ${f}_locked.jsonl(.gz) in $SRC"; ls -la "$SRC"; exit 1; fi
done
diff "$LOCK" atml-pa4/splits/data_lock/local_source_revisions.json && echo "LOCK MATCHES"
ls -la atml-pa4-datasets/data/public_locked
echo "code at $(git -C atml-pa4 rev-parse --short HEAD) on $BRANCH"
python -c "import torch, transformers, scipy; print(torch.__version__, transformers.__version__, scipy.__version__)"

In [ ]:
%%bash
set -e
cd /tmp/pa4/atml-pa4
python -m pytest tests -q 2>&1 | tail -3

In [ ]:
%%bash
cd /tmp/pa4/atml-pa4
RES=/kaggle/working/results/task1
SPL=/kaggle/working/task1_splits
mkdir -p "$RES/logs" "$SPL"
rm -rf splits/task1 && ln -s "$SPL" splits/task1
COMMON="--data_root /tmp/pa4/atml-pa4-datasets --output_dir /tmp/pa4/outputs"

run() {
  tag=$1; shift
  PYTHONUNBUFFERED=1 python -m "$@" $COMMON 2>&1 | grep --line-buffered -v -i "warn" | tee -a "$RES/logs/$tag.log" | sed -u "s/^/[$tag] /"
  return ${PIPESTATUS[0]}
}

#calibration-only sensitivity, then q3 allocations (cpu ilp): all frozen before any held-out eval
CUDA_VISIBLE_DEVICES=0 run adaptive task1.sensitivity || exit 1
run adaptive task1.allocate --method q3 --bits 4 --results_dir "$RES" || exit 1
run adaptive task1.allocate --method q3 --bits 3 --results_dir "$RES" || exit 1

chain() {
  gpu=$1; s=$2; b=$3
  export CUDA_VISIBLE_DEVICES=$gpu
  run gpu$gpu task1.allocate --method p3 --sparsity $s --results_dir "$RES" &&
  run gpu$gpu task1.run_condition --method p3_nonuniform --sparsity $s --evals $EVALS --results_dir "$RES" &&
  run gpu$gpu task1.run_condition --method q3_mixed_budget --bits $b --evals $EVALS --results_dir "$RES"
}

chain 0 0.5 4 & A=$!
chain 1 0.75 3 & B=$!
wait $A; ra=$?
wait $B; rb=$?
echo "gpu0 chain exit $ra, gpu1 chain exit $rb"
ls -R "$SPL"

In [ ]:
import json, glob, os
RES= "/kaggle/working/results/task1"
SPL= "/kaggle/working/task1_splits"
for m in NAMES:
    print(f"--- {m} ---")
    d= None
    for e in EVALS.split():
        p= os.path.join(RES, m, f"{e}.json")
        if not os.path.exists(p):
            print(e, "MISSING")
            continue
        d= json.load(open(p))
        if e in ["wikitext2", "c4_val"]:
            print(f"{e}: ppl={d['ppl']:.4f} n={d['n_scored']} windows={d['n_windows']} time={d['wall_clock_seconds']:.0f}s")
        elif e == "hellaswag":
            print(f"hellaswag: acc={d['acc']:.4f} acc_norm={d['acc_norm']:.4f} n={d['n_examples']} time={d['wall_clock_seconds']:.0f}s")
        else:
            for k, v in d["latency"].items():
                print(f"{k}: median={v['median']:.1f} ms iqr={v['iqr']:.1f} peak_alloc={v['peak_allocated_bytes']/2**30:.2f} GiB")
            print("storage:", d["storage"])
    c= os.path.join(RES, m, "condition.json")
    if os.path.exists(c):
        c= json.load(open(c))
        print("build:", {k: c[k] for k in c if k in ["total_build_seconds", "matched_zero_check", "frozen_allocation"]})
    if d:
        print("commit:", d["git_commit"], "gpu:", d["gpu_name"], "dtype:", d["dtype"], "sdpa:", d["sdpa"])

p= os.path.join(SPL, "sensitivity.json")
if os.path.exists(p):
    d= json.load(open(p))
    print(f"--- sensitivity: {len(d['units'])} units, {d['wall_clock_seconds']:.0f}s, loss-scale halvings {d['loss_scale_halvings']}, peak {d['peak_allocated_bytes']/2**30:.2f} GiB")
for f in sorted(glob.glob(os.path.join(SPL, "p3_allocation_*.json"))):
    d= json.load(open(f))
    print(os.path.basename(f), "chosen lambda", d["chosen_lambda"], "zeros", d["total_zeros"])
    for r in d["dev_sweep"]:
        print(f"  lam {r['lambda']:g}: dev nll {r['dev_mean_nll']:.5f} ppl {r['dev_ppl']:.3f}")
for f in sorted(glob.glob(os.path.join(SPL, "q3_allocation_*.json"))):
    d= json.load(open(f))
    print(os.path.basename(f), d["bit_counts"], d["packed_total_bytes"], "<=", d["budget_total_bytes"], f"objective {d['objective']:.4g} vs uniform {d['objective_uniform']:.4g}")
for f in sorted(glob.glob(os.path.join(RES, "logs", "*.log"))):
    print(f"--- last lines of {os.path.basename(f)} ---")
    print("".join(open(f).readlines()[-5:]))